**Цель проекта - реализовать инструктивное дообучение малой языковой модели методом QLoRA для решения задачи Text-to-SQL, то есть генерации SQL-запросов на основе текстового описания пользователя и схемы базы данных**

Разбираемся со Spider

После скачивания оно примерно в таким виде лежит: 
```text
spider/
├── train_spider.json
├── train_others.json
├── dev.json
├── tables.json
└── database/
    ├── academic/
    │   ├── academic.sqlite
    │   └── schema.sql
    ├── activity_1/
    │   ├── activity_1.sqlite
    │   └── schema.sql
    └── ...
```

Что нам из этого всего добра нужно вытащить
```text
train_spider.json — основной train
train_others.json — дополнительный train, можно добавить
dev.json — наш бенчмарк
tables.json — схемы БД
database/ — реальные SQLite-базы для execution accuracy
```

In [4]:
import json
from pathlib import Path

In [7]:
# pip install torch transformers datasets peft trl bitsandbytes accelerate

In [5]:
spider_dir = Path("spider")

for name in ["train_spider.json", "train_others.json", "dev.json", "tables.json"]:
    path = spider_dir / name
    print(name, "exists:", path.exists())

with open(spider_dir / "train_spider.json", encoding="utf-8") as f:
    train_spider = json.load(f)

with open(spider_dir / "train_others.json", encoding="utf-8") as f:
    train_others = json.load(f)

with open(spider_dir / "dev.json", encoding="utf-8") as f:
    dev = json.load(f)

with open(spider_dir / "tables.json", encoding="utf-8") as f:
    tables = json.load(f)

print("train_spider:", len(train_spider))
print("train_others:", len(train_others))
print("dev:", len(dev))
print("databases in tables.json:", len(tables))
print("database folder exists:", (spider_dir / "database").exists())

train_spider.json exists: True
train_others.json exists: True
dev.json exists: True
tables.json exists: True
train_spider: 7000
train_others: 1659
dev: 1034
databases in tables.json: 166
database folder exists: True


In [9]:
with open("processed_spider_sft/train_sft.jsonl", encoding="utf-8") as f:

    example = json.loads(next(f))

print(example["text"])

### Instruction:
You are a Text-to-SQL assistant. Generate a valid SQLite SQL query for the given question using only the provided database schema. Do not use tables or columns that are not present in the schema. Return only the SQL query.

### Database schema:
CREATE TABLE department (
    Department_ID NUMBER,
    Name TEXT,
    Creation TEXT,
    Ranking NUMBER,
    Budget_in_Billions NUMBER,
    Num_Employees NUMBER,
    PRIMARY KEY (Department_ID)
);

CREATE TABLE head (
    head_ID NUMBER,
    name TEXT,
    born_state TEXT,
    age NUMBER,
    PRIMARY KEY (head_ID)
);

CREATE TABLE management (
    department_ID NUMBER,
    head_ID NUMBER,
    temporary_acting TEXT,
    PRIMARY KEY (department_ID),
    FOREIGN KEY (head_ID) REFERENCES head(head_ID),
    FOREIGN KEY (department_ID) REFERENCES department(Department_ID)
);

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT count(*) FROM head WHERE age > 56


In [10]:
with open("processed_spider_sft/dev_eval_prompts.jsonl", encoding="utf-8") as f:

    example = json.loads(next(f))

print(example["text"])

print("REFERENCE SQL:", example["sql"])

### Instruction:
You are a Text-to-SQL assistant. Generate a valid SQLite SQL query for the given question using only the provided database schema. Do not use tables or columns that are not present in the schema. Return only the SQL query.

### Database schema:
CREATE TABLE stadium (
    Stadium_ID NUMBER,
    Location TEXT,
    Name TEXT,
    Capacity NUMBER,
    Highest NUMBER,
    Lowest NUMBER,
    Average NUMBER,
    PRIMARY KEY (Stadium_ID)
);

CREATE TABLE singer (
    Singer_ID NUMBER,
    Name TEXT,
    Country TEXT,
    Song_Name TEXT,
    Song_release_year TEXT,
    Age NUMBER,
    Is_male OTHERS,
    PRIMARY KEY (Singer_ID)
);

CREATE TABLE concert (
    concert_ID NUMBER,
    concert_Name TEXT,
    Theme TEXT,
    Stadium_ID TEXT,
    Year TEXT,
    PRIMARY KEY (concert_ID),
    FOREIGN KEY (Stadium_ID) REFERENCES stadium(Stadium_ID)
);

CREATE TABLE singer_in_concert (
    concert_ID NUMBER,
    Singer_ID TEXT,
    PRIMARY KEY (concert_ID),
    FOREIGN KEY (Singer_ID) REF

Короче вроде все красиво. Так нами для обучения модели был подготовлен датасет на основе Spider. Для каждого примера по db_id извлекалась схема базы данных из tables.json, после чего она преобразовывалась в формат CREATE TABLE с указанием первичных и внешних ключей. Вход модели включал инструкцию, схему базы данных и вопрос на естественном языке, а целевым ответом являлся SQL-запрос. Такой формат позволяет обучать модель генерировать SQL с учётом структуры конкретной базы данных

train_sft.jsonl — основной обучающий файл для QLoRA.
В каждой строке один пример: инструкция, схема БД, вопрос и правильный SQL-ответ.

dev_sft.jsonl — dev-часть в таком же формате, как трейн.
Её можно использовать для подсчёта validation loss во время обучения.

dev_eval_prompts.jsonl — файл для финальной генерации и оценки.
В поле text есть инструкция, схема и вопрос, но нет правильного SQL. Эталонный SQL хранится отдельно в поле sql.

train_preview_100.json — первые 100 обучающих примеров в удобном JSON-формате.
Нужен, чтобы глазами проверить, правильно ли собрался датасет.

dev_preview_100.json — первые 100 dev-примеров.
Нужен для ручной проверки dev-части.

stats.json — статистика собранного датасета.
Там хранится количество train/dev примеров, число уникальных баз данных и пути к созданным файлам.